# Traitement des fichiers bb_rr par blocs V4

In [ ]:
from scipy import signal, interpolate
import os
import pandas as pd
import numpy as np

In [1]:
SEUIL_RR_VALIDES = 400

dossier_rr = r"C:\Users\judupont\Desktop\bb_rr_tronque_blocs_v4"
dossier_sortie = r"C:\Users\judupont\Desktop\RtoR_v4"
os.makedirs(dossier_sortie, exist_ok=True)

# ======================================================
# FONCTION TRAITEMENT RtoR → RR
# ======================================================

def traiter_bloc_rtor(
    df_bloc,
    window_size=300,
    min_rr=300,
    max_rr=2000,
    sd_factor=2
):
    rr_ms = []
    timestamps_rr = []

    valeur_rr_precedente = None
    signe_precedent = None
    timestamp_premiere_apparition = None

    for _, row in df_bloc.iterrows():
        valeur_rr = float(row["RtoR"])
        timestamp = row["Timestamp"]

        if valeur_rr == 0:
            continue

        signe_actuel = 1 if valeur_rr >= 0 else -1

        if signe_precedent is not None and signe_actuel != signe_precedent:
            valeur_ms = abs(valeur_rr_precedente) * 1000
            if min_rr <= valeur_ms <= max_rr:
                rr_ms.append(int(valeur_ms))
                timestamps_rr.append(timestamp_premiere_apparition)
            timestamp_premiere_apparition = timestamp

        elif signe_precedent is None:
            timestamp_premiere_apparition = timestamp

        valeur_rr_precedente = valeur_rr
        signe_precedent = signe_actuel

    # dernier RR
    if valeur_rr_precedente is not None and timestamp_premiere_apparition is not None:
        valeur_ms = abs(valeur_rr_precedente) * 1000
        if min_rr <= valeur_ms <= max_rr:
            rr_ms.append(int(valeur_ms))
            timestamps_rr.append(timestamp_premiere_apparition)

    df_rr = pd.DataFrame({
        "Timestamp_RR": timestamps_rr,
        "rr_ms": rr_ms
    })

    if len(df_rr) < window_size:
        df_rr["rr_ms_filt"] = np.nan
        return df_rr

    rr = df_rr["rr_ms"].values
    indices_to_remove = set()

    for i in range(len(rr)):
        if i + window_size <= len(rr):
            window = rr[i:i + window_size]
            start_idx = i
        else:
            window = rr[-window_size:]
            start_idx = len(rr) - window_size

        mean_w = np.mean(window)
        std_w = np.std(window, ddof=1)
        lower = mean_w - sd_factor * std_w
        upper = mean_w + sd_factor * std_w

        for j in range(window_size):
            idx = start_idx + j
            if idx < len(rr):
                if rr[idx] < lower or rr[idx] > upper:
                    indices_to_remove.add(idx)

    rr_filt = rr.astype(float)
    rr_filt[list(indices_to_remove)] = np.nan
    df_rr["rr_ms_filt"] = rr_filt

    return df_rr


# ======================================================
# RECUPERATION
# ======================================================

fichiers = sorted([f for f in os.listdir(dossier_rr) if f.endswith(".csv")])
print(f"📂 {len(fichiers)} fichiers détectés\n")

resultats_temp = []
blocs_supprimes = []

# =========================================
# BOUCLE SUR LES FICHIERS 
# =========================================
for i, fichier in enumerate(fichiers, 1):
    print(f"[{i}/{len(fichiers)}] Traitement {fichier}")

    try:
        df_bloc = pd.read_csv(os.path.join(dossier_rr, fichier))

        pid, bloc = fichier.replace(".csv", "").split("_", 1)

        df_rr = traiter_bloc_rtor(df_bloc)
        n_rr_valides = df_rr["rr_ms_filt"].notna().sum()
        valide = n_rr_valides >= SEUIL_RR_VALIDES

        print(f"   RR valides : {n_rr_valides} → {'OK' if valide else 'REJET'}")

        resultats_temp.append({
            "pid": pid,
            "bloc": bloc,
            "fichier": fichier,
            "df_rr": df_rr,
            "n_rr_valides": n_rr_valides,
            "valide": valide
        })

        if not valide:
            blocs_supprimes.append({
                "Patient": pid,
                "Bloc": bloc,
                "Raison": f"RR valides < {SEUIL_RR_VALIDES}",
                "RR_valides": n_rr_valides
            })

    except Exception as e:
        print(f"   ❌ Erreur : {e}")

# =========================================================
# SI UN BLOC MANQUE ON REJETTE TOUS LES BLOCS DU CANDIDAT
# =========================================================

print("\n==============================")
print("🔍 FILTRAGE FINAL PAR PATIENT (TOUT OU RIEN)")
print("==============================")

patients = sorted(set(r["pid"] for r in resultats_temp))
lignes_qc = []

for pid in patients:
    blocs = [r for r in resultats_temp if r["pid"] == pid]

    blocs_invalides = [b for b in blocs if not b["valide"]]

    if blocs_invalides:
        print(f"❌ {pid} → {len(blocs_invalides)} bloc(s) manquant(s)/invalide(s) → SUPPRESSION TOTALE")

        for b in blocs:
            blocs_supprimes.append({
                "Patient": pid,
                "Bloc": b["bloc"],
                "Raison": "Au moins un bloc manquant ou invalide",
                "RR_valides": b["n_rr_valides"]
            })
        continue

    print(f"✅ {pid} → tous les blocs valides → sauvegarde")

    for b in blocs:
        nom_sortie = b["fichier"].replace(".csv", "_RtoR_v4.csv")
        b["df_rr"].to_csv(
            os.path.join(dossier_sortie, nom_sortie),
            index=False
        )

        lignes_qc.append({
            "Patient": pid,
            "Bloc": b["bloc"],
            "RR_valides": b["n_rr_valides"]
        })


# ======================================================
# LISTE DES BLOCS SUPPRIMES
# ======================================================

print("\n==============================")
print("🧹 BLOCS SUPPRIMÉS (DÉTAIL)")
print("==============================")

if blocs_supprimes:
    df_supprimes = pd.DataFrame(blocs_supprimes)
    df_supprimes = df_supprimes.sort_values(["Patient", "Bloc"])
    print(df_supprimes.to_string(index=False))
else:
    print("✅ Aucun bloc supprimé")

# ======================================================
# DTAFRAME AVEC CANDIDATS CONSERVES
# ======================================================

df_qc = pd.DataFrame(lignes_qc)
print("\n==============================")
print("📊 QC FINAL (BLOCS CONSERVÉS)")
print("==============================")
print(df_qc)  

print("\n🏁 Traitement terminé")


📂 402 fichiers détectés

[1/402] Traitement 0101CAR_bloc1.csv
   RR valides : 0 → REJET
[2/402] Traitement 0101CAR_bloc2.csv
   RR valides : 0 → REJET
[3/402] Traitement 0101CAR_bloc3.csv
   RR valides : 0 → REJET
[4/402] Traitement 0101EMS_bloc1.csv
   RR valides : 2768 → OK
[5/402] Traitement 0101EMS_bloc2.csv
   RR valides : 2491 → OK
[6/402] Traitement 0101EMS_bloc3.csv
   RR valides : 2760 → OK
[7/402] Traitement 0102EMS_bloc1.csv
   RR valides : 539 → OK
[8/402] Traitement 0102EMS_bloc2.csv
   RR valides : 1598 → OK
[9/402] Traitement 0102EMS_bloc3.csv
   RR valides : 2302 → OK
[10/402] Traitement 0102PCR_bloc1.csv
   RR valides : 841 → OK
[11/402] Traitement 0102PCR_bloc2.csv
   RR valides : 3052 → OK
[12/402] Traitement 0102PCR_bloc3.csv
   RR valides : 1428 → OK
[13/402] Traitement 0103BPS_bloc1.csv
   RR valides : 1509 → OK
[14/402] Traitement 0103BPS_bloc2.csv
   RR valides : 1742 → OK
[15/402] Traitement 0103BPS_bloc3.csv
   RR valides : 1964 → OK
[16/402] Traitement 0104FJ

# STAT METRIQUE

In [13]:
def hrv_time_domain(rr_ms):
    if len(rr_ms) < 250:
        return {}
    diff_rr = np.diff(rr_ms)
    return {
        "Mean_RR_ms": np.mean(rr_ms),
        "Mean_HR_bpm": 60000 / np.mean(rr_ms),
        "SDNN_ms": np.std(rr_ms, ddof=1),
        "RMSSD_ms": np.sqrt(np.mean(diff_rr ** 2)),
        "NN50": np.sum(np.abs(diff_rr) > 50),
        "pNN50_%": 100 * np.sum(np.abs(diff_rr) > 50) / len(diff_rr)
    }
def hrv_frequency_domain(rr_ms):
    if len(rr_ms) < 250:
        return {}

    time_ms = np.cumsum(rr_ms)
    time_ms = np.insert(time_ms, 0, 0)[:-1]

    fs = 4.0
    time_interp = np.arange(time_ms[0], time_ms[-1], 1000 / fs)

    f_interp = interpolate.interp1d(
        time_ms, rr_ms, kind="cubic", fill_value="extrapolate"
    )
    rr_interp = f_interp(time_interp)

    rr_detrend = signal.detrend(rr_interp)
    nperseg = min(256, len(rr_detrend))

    freqs, psd = signal.welch(
        rr_detrend,
        fs=fs,
        window="hann",
        nperseg=nperseg,
        scaling="density"
    )

    def band_power(low, high):
        idx = (freqs >= low) & (freqs < high)
        return np.trapezoid(psd[idx], freqs[idx])

    vlf = band_power(0.0033, 0.04)
    lf  = band_power(0.04, 0.15)
    hf  = band_power(0.15, 1)

    return {
        "VLF_power_ms2": vlf,  # Very Low Frequency
        "LF_power_ms2": lf,    # Low Freqency
        "HF_power_ms2": hf,    # High Frequency
        "LF_HF_ratio": lf / hf if hf > 0 else np.nan
    }

In [10]:
def calculer_metriques_bloc(df_bloc):
    if "rr_ms_filt" not in df_bloc.columns:
        return None

    rr_ms = df_bloc["rr_ms_filt"].dropna().values

    if len(rr_ms) < 250:
        return None

    metriques = {}
    metriques.update(hrv_time_domain(rr_ms))
    metriques.update(hrv_frequency_domain(rr_ms))

    metriques["RR_conserves"] = len(rr_ms)
    metriques["RR_initiaux"] = df_bloc["rr_ms"].notna().sum()
    metriques["Pct_conserve"] = (
        100 * metriques["RR_conserves"] / max(metriques["RR_initiaux"], 1)
    )

    return metriques

In [12]:
dossier_rr = r"C:\Users\judupont\Desktop\RtoR_v4"
chemin_sortie = r"C:\Users\judupont\Desktop\HRV_RtoR_V4_python.csv"
chemin_df_global_txt = r"C:\Users\judupont\Desktop\df_global_v4.txt"

# =================================================================
# CHARGEMENT FICHIER TXT POUR RECUPERER LA CONDITION ET LA FEUILLE 
# =================================================================
df_global = pd.read_csv(chemin_df_global_txt, sep="\t", encoding="utf-8")

# ======================================================
# LISTE DES FICHIERS
# ======================================================
fichiers = sorted([f for f in os.listdir(dossier_rr) if f.endswith(".csv")])
print(f"📂 {len(fichiers)} fichiers RR détectés\n")

resultats = []

# ======================================================
# BOUCLE SUR LES FICHIERS
# ======================================================
for i, fichier in enumerate(fichiers, 1):
    chemin = os.path.join(dossier_rr, fichier)
    print(f"[{i}/{len(fichiers)}] Calcul HRV : {fichier}")

    try:
        df = pd.read_csv(chemin)

        if "rr_ms_filt" not in df.columns:
            print("   ❌ Colonne rr_ms_filt absente → ignoré\n")
            continue

        metriques = calculer_metriques_bloc(df)
        if metriques is None:
            print("   ⚠️ Pas assez de RR filtrés → ignoré\n")
            continue

        # Extraction Patient / Bloc
        nom = fichier.replace("_RtoR_v4.csv", "")
        pid, bloc = nom.split("_", 1)   # ex : bloc1

        metriques["Numero_inclusion"] = pid
        metriques["Bloc"] = bloc.lower()  # bloc1, bloc2, bloc3

        resultats.append(metriques)
        print("   ✅ OK\n")

    except Exception as e:
        print(f"   ❌ ERREUR : {e}\n")

# ======================================================
# DATAFRAME LONG (1 LIGNE = 1 BLOC)
# ======================================================
df_resultats = pd.DataFrame(resultats)

# ====================================================================
# AJOUT COLONNE FEUILLE : CORRESPOND AU NOM DU SERVICE (APHM,CEAN...)
# ====================================================================
df_resultats = df_resultats.merge(
    df_global[["Numero_inclusion", "Feuille"]],
    on="Numero_inclusion",
    how="left"
)

# ======================================================
# PASSAGE EN FORMAT WIDE : (1 LIGNE PAR PATIENT)
# ======================================================
print("🔄 Conversion en format WIDE")

stats_bloc = [
    "Mean_RR_ms",
    "Mean_HR_bpm",
    "SDNN_ms",
    "RMSSD_ms",
    "pNN50_%",
    "VLF_power_ms2",
    "LF_power_ms2",
    "HF_power_ms2",
    "LF_HF_ratio",
]

rows = []

for patient, df_p in df_resultats.groupby("Numero_inclusion"):
    ligne = {
        "Feuille": df_p["Feuille"].iloc[0],
        "Numero_inclusion": patient,
    }
    pct_blocs = []  # pour stocker les Pct_conserve de chaque bloc
    
    for bloc in ["bloc1", "bloc2", "bloc3"]:
        df_b = df_p[df_p["Bloc"] == bloc]

        # Nom du bloc écrit dans la cellule
        ligne[f"{bloc.capitalize()}"] = bloc if not df_b.empty else ""

        for stat in stats_bloc:
            col = f"{stat}_{bloc.capitalize()}"
            if not df_b.empty and stat in df_b.columns:
                ligne[col] = df_b.iloc[0][stat]
            else:
                ligne[col] = np.nan

        # Stocker Pct_conserve pour ce bloc pour calculer la moyenne finale
        if not df_b.empty and "Pct_conserve" in df_b.columns:
            pct_blocs.append(df_b.iloc[0]["Pct_conserve"])
        else:
            pct_blocs.append(np.nan)

    # ======================================================
    # MOYENNE DE Pct_conserve SUR LES 3 BLOCS
    # ======================================================
    ligne["Pct_conserve"] = np.nanmean(pct_blocs)

    rows.append(ligne)

df_final = pd.DataFrame(rows)

# ======================================================
# ORDRE DES COLONNES FINAL DANS LE CSV
# ======================================================
ordre_colonnes = [
    "Numero_inclusion",
    "Feuille",

    "Mean_RR_ms_Bloc1",
    "Mean_HR_bpm_Bloc1",
    "SDNN_ms_Bloc1",
    "RMSSD_ms_Bloc1",
    "pNN50_%_Bloc1",
    "VLF_power_ms2_Bloc1",
    "LF_power_ms2_Bloc1",
    "HF_power_ms2_Bloc1",
    "LF_HF_ratio_Bloc1",

    "Mean_RR_ms_Bloc2",
    "Mean_HR_bpm_Bloc2",
    "SDNN_ms_Bloc2",
    "RMSSD_ms_Bloc2",
    "pNN50_%_Bloc2",
    "VLF_power_ms2_Bloc2",
    "LF_power_ms2_Bloc2",
    "HF_power_ms2_Bloc2",
    "LF_HF_ratio_Bloc2",

    "Mean_RR_ms_Bloc3",
    "Mean_HR_bpm_Bloc3",
    "SDNN_ms_Bloc3",
    "RMSSD_ms_Bloc3",
    "pNN50_%_Bloc3",
    "VLF_power_ms2_Bloc3",
    "LF_power_ms2_Bloc3",
    "HF_power_ms2_Bloc3",
    "LF_HF_ratio_Bloc3",

    "Pct_conserve",  
]

ordre_colonnes = [c for c in ordre_colonnes if c in df_final.columns]
df_final = df_final[ordre_colonnes]

df_final = df_final.sort_values(by="Pct_conserve", ascending=False).reset_index(drop=True)
df_final = df_final.round(2)

# ======================================================
# SAUVEGARDE
# ======================================================

df_final.to_csv(chemin_sortie, index=False, encoding="utf-8-sig")
print(f"\n💾 Résultats sauvegardés : {chemin_sortie}")
print("🏁 Calcul HRV terminé")

📂 282 fichiers RR détectés

[1/282] Calcul HRV : 0101EMS_bloc1_RtoR_v4.csv
   ✅ OK

[2/282] Calcul HRV : 0101EMS_bloc2_RtoR_v4.csv
   ✅ OK

[3/282] Calcul HRV : 0101EMS_bloc3_RtoR_v4.csv
   ✅ OK

[4/282] Calcul HRV : 0102EMS_bloc1_RtoR_v4.csv
   ✅ OK

[5/282] Calcul HRV : 0102EMS_bloc2_RtoR_v4.csv
   ✅ OK

[6/282] Calcul HRV : 0102EMS_bloc3_RtoR_v4.csv
   ✅ OK

[7/282] Calcul HRV : 0102PCR_bloc1_RtoR_v4.csv
   ✅ OK

[8/282] Calcul HRV : 0102PCR_bloc2_RtoR_v4.csv
   ✅ OK

[9/282] Calcul HRV : 0102PCR_bloc3_RtoR_v4.csv
   ✅ OK

[10/282] Calcul HRV : 0103BPS_bloc1_RtoR_v4.csv
   ✅ OK

[11/282] Calcul HRV : 0103BPS_bloc2_RtoR_v4.csv
   ✅ OK

[12/282] Calcul HRV : 0103BPS_bloc3_RtoR_v4.csv
   ✅ OK

[13/282] Calcul HRV : 0104IBS_bloc1_RtoR_v4.csv
   ✅ OK

[14/282] Calcul HRV : 0104IBS_bloc2_RtoR_v4.csv
   ✅ OK

[15/282] Calcul HRV : 0104IBS_bloc3_RtoR_v4.csv
   ✅ OK

[16/282] Calcul HRV : 0107DSS_bloc1_RtoR_v4.csv
   ✅ OK

[17/282] Calcul HRV : 0107DSS_bloc2_RtoR_v4.csv
   ✅ OK

[18/282] Cal